# Spark Functions Explained with a Salary Dataset

In this notebook, we will explain the key functions used in the [Airbnb pipeline project](airbnb_pipeline_project.ipynb), demonstrated with a simple dataset about people and their salary. The dataset contains information about people's salary, department, and years of experience.

## 1. Loading a CSV File into a Spark DataFrame

The first step in any Spark analysis is to load the data. We read the data directly from the GitHub repository of the course, so there is nothing to upload. pandas downloads the CSV and Spark turns it into a DataFrame. This is fine for small files; with real big data Spark would read the files directly from distributed storage (see [Lesson 7](https://github.com/Naviden/Introduction-to-Big-Data-and-NoSQL/blob/main/lessons/07_spark_basics_part1.md) for the other options).

In [ ]:
import os
from pyspark.sql import SparkSession

# On Databricks this returns the session that already exists.
# Locally (pip install pyspark) or on Colab it starts a new one.
spark = SparkSession.builder.getOrCreate()

# display(df) shows an interactive table only on Databricks: elsewhere we use show() instead
if "DATABRICKS_RUNTIME_VERSION" not in os.environ:
    def display(df):
        df.show(20, truncate=False)

In [ ]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/Naviden/Introduction-to-Big-Data-and-NoSQL/main/data"

def load_csv(path, **kwargs):
    """Read a CSV file of the course repository and return a Spark DataFrame."""
    pdf = pd.read_csv(f"{DATA_URL}/{path}", **kwargs)
    pdf = pdf.astype(object).where(pdf.notna(), None)  # pandas NaN -> Spark null
    return spark.createDataFrame(pdf)

In [ ]:
# Load the CSV file into a Spark DataFrame
df = load_csv("misc/salary_data.csv")

# Show the first few rows of the DataFrame
df.show()

If you uploaded the file to a volume in Databricks instead, you would read it like this:

```python
df = spark.read.csv("/Volumes/workspace/default/course_data/salary_data.csv", header=True, inferSchema=True)
```

## 2. Filtering Data

The `filter()` function is used to filter rows based on a condition. For example, we can filter people whose salary is greater than $60,000.

In [ ]:
# Filter rows where salary is greater than 60,000
high_salary_df = df.filter(df.salary > 60000)
high_salary_df.show()

## 3. Adding a New Column

We can use the `withColumn()` function to add a new column. In this case, we'll add a column that categorizes people into salary tiers: Low, Medium, and High.

In [ ]:
from pyspark.sql.functions import when

# Add a new column 'salary_tier' based on the salary
df = df.withColumn("salary_tier",
                   when(df.salary < 60000, "Low")
                   .when((df.salary >= 60000) & (df.salary <= 70000), "Medium")
                   .otherwise("High"))

df.show()

## 4. Aggregating Data

The `groupBy()` function is used to group data by one or more columns. We can then use aggregation functions like `avg()` to calculate the average salary in each department.

In [ ]:
# Group data by 'department' and calculate the average salary
avg_salary_by_department = df.groupBy("department").avg("salary")
avg_salary_by_department.show()

## 5. Data Cleaning (Handling Null Values)

To clean the data, we can use functions like `fillna()` to replace `null` values with a default value. For example, if the 'experience_years' column has missing values, we can fill them with 0.

This small dataset has no missing values, so we first add a row with a missing value to see the effect.

In [ ]:
from pyspark.sql import Row

with_missing = df.unionByName(spark.createDataFrame(
    [Row(person_id=6, name="Frank", department="IT", salary=65000, experience_years=None, salary_tier="Medium")],
    schema=df.schema,
))
with_missing.show()

# Fill missing values in 'experience_years' with 0
cleaned_df = with_missing.fillna({'experience_years': 0})
cleaned_df.show()